# RAG Bot 

The RAG bot system to answer the question about Ethnicity in Thailand.

## Setting Up   

### Install Libraries & Dependencies  

In [ ]:
%pip install python-dotenv easyocr pypdf  

### Import Libraries & Dependencies  

In [ ]:
import os, re, json, glob   
import time  
import requests  
from dotenv import load_dotenv   

### Setting Environments  

In [ ]:
load_dotenv()

In [ ]:
api_key = os.getenv("THAILLM_API_KEY") 

if api_key:
  print(f"[LOG] Success: Get API Key success.")
else:
  print(f"[LOG] Fail: Cannot Get API Key.")  

## LLM Implementation  

In [ ]:
def ask(messages: str, model="typhoon", max_retries=5):
  url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"  
  headers = {
    "Content-Type": "application/json",
    "apikey": api_key  
  }
  payload = {
    "model": "/model",
    "messages": messages,
    "max_tokens": 2024,
    "temperature": 0 
  }

  for attempt in range(max_retries):
    try:
      resp = requests.post(url=url, headers=headers, json=payload, timeout=120)

      if resp.status_code == 429:
        wait = min(2 ** attempt, 30)
        print(f"Rate Limit, wating {wait} sec(s).")
        time.sleep(wait)
        continue 
      
      resp.raise_for_status()
      text_answer = resp.json()["choices"][0]["message"]["content"].strip() 
      clean_answer = re.sub(r"<think>.*?</think>", "", text_answer, flags=re.DOTALL).strip()
      return clean_answer    
    except requests.exceptions.RequestException as e:
      wait = 2 ** attempt
      print(f"Error: {e}, Retrying in {wait} sec(s).")
      time.sleep(wait)
    
  return None  


In [ ]:
def qa(question: str):
  prompt = {"role": "user", "content": question}
  
  print(f"[LOG] Process for the Question: {question}")

  answer_typhoon = ask([prompt], model="typhoon")
  answer_kbtg = ask([prompt], model="kbtg")
  answer_openthaigpt = ask([prompt], model="openthaigpt")
  answer_pathumma = ask([prompt], model="pathumma")

  # JSON Format
  # answer = {
  #   "Typhoon": f"[RAG BOT (Typhoon)] {answer_typhoon}",
  #   "KBTG": f"[RAG BOT (KBTG)] {answer_kbtg}",
  #   "OpenThaiGPT": f"[RAG BOT (OpenThaiGPT)] {answer_openthaigpt}",
  #   "Pathumma": f"[RAG BOT (Pathumma)] {answer_pathumma}"
  # }

  # Markdown Format   
  answer = f"""# Question
{question}

# Answer

## Typhoon
{answer_typhoon}

## KBTG
{answer_kbtg}

## OpenThaiGPT
{answer_openthaigpt}

## Pathumma
{answer_pathumma} 
"""

  return answer   

### LLM Implementation Testing  

In [ ]:
output_path = "../output/answer/markdown"
def report(question: str, filename: str, output_path_dir: str = f"{output_path}"):
  # question = "คนไทยเชื้อสายจีนในเมืองหาดใหญ่ ส่วนมากเป็นคนเชื้อสายจีนสายไหน"
  clean_question = " ".join(question.split(" ")[1:])  
  result = qa(clean_question)

  # output = {
  #   "question": question,
  #   "answer": test_response_01
  # }

  # output_path = "../output/answer/markdown"  

  if filename:
    file_name = filename
  else:
    file_name = "_".join(question.split(" "))

  with open(f"{output_path_dir}/{file_name}.md", "w", encoding="utf-8") as file:
    file.write(result)  

In [ ]:
file_name = "QA_Hainanese"
if os.path.isfile(f"{output_path}/{file_name}.md"):
  print(f"[LOG] Already have file.")
else:
  report("คนไทยเชื้อสายจีนไหหลำในไทย อาศัยอยู่กันเยอะในจังหวัดอะไรในประเทศไทยบ้าง", filename="QA_Hainanese")  

In [ ]:
file_name = "QA_Vietnamese"
if os.path.isfile(f"{output_path}/{file_name}.md"):
  print(f"[LOG] Already have file.")
else:  
  report("คนไทยญวน พูดภาษาเวียดนามเหมือนกับที่เวียดนามในปัจจุบันพูดหรือไม่", filename="QA_Vietnamese")    

In [ ]:
with open("../data/questions/questions.txt", "r", encoding="utf-8") as f:
  questions = f.readlines()

for index, question in enumerate(questions):
  output_dir = "../output/answer/markdown/question_txt_response/llm_without_rag"
  
  # File Name Setting.
  if index+1 >= 0 or index+1 <= 9:
    file_name = f"answer_00{index+1}"
  elif index+1 >= 10 or index+1 <= 99:
    file_name = f"answer_0{index+1}"
  else:
    file_name = f"answer_{index+1}" 
  
  # Check If already has file for the question. 
  if os.path.isfile(f"{output_dir}/{file_name}"):
    print(f"[LOG] Already has file: {file_name}")
  else:
    report(question.strip(), filename=file_name, output_path_dir=output_dir)  

## RAG Implementation  

### PDF OCR to Markdown   

In [ ]:
knowledge = "../data/knowledge/pdf"
files = glob.glob(f"{knowledge}/*.pdf") 
print(files)  
print(f"[LOG] Amount of File: {len(files)} files.")  

In [ ]:
from pypdf import PdfReader  

for index, file in enumerate(sorted(files)):
  extract_data = ""
  reader = PdfReader(file)
  text = f"\n".join(p.extract_text() for p in reader.pages) 
  extract_data += text 
  with open(f"../data/knowledge/text/{file.split("/")[-1].split(".")[0]}.txt", "w", encoding="utf-8-sig") as f:
    f.write(extract_data)

## Add Documents  

In [ ]:
import os  

In [ ]:
def get_all_documents(data_path: str):
  documents = []
  file_names = []

  for filename in os.listdir(data_path):
    if filename.endswith(".txt"):
      file_names.append(filename.split(".")[0])
      file_path = f"{data_path}/{filename}"
      with open(file_path, "r", encoding="utf-8") as f:
        documents.append(f.read())
  
  return documents, file_names  

In [ ]:
knowledge_dir_path = "../data/knowledge/text"

docs, fs = get_all_documents(knowledge_dir_path)

print(f"File: {fs}")